# Maintenance Rehearsal (A) — Data Prep

- **input** = the source article, truncated to `configs/chunking.yaml`'s
  `max_words` (350 words) — this matters: `rehearse_maintenance` (see
  `src/pipeline/rehearsal.py`) always operates on a single *chunk* (~350
  words), never a whole raw document, so the training input needs to be the
  same scale as what the model will actually see at inference time.
- **target** = an approximate extractive summary built via **greedy ROUGE
  oracle extraction** over that same truncated text (see §2 below for why).

This replaces the original Korean pipeline's AIHub document-summarization
corpus, which shipped with explicit extractive sentence-index labels. English
`cnn_dailymail` only has abstractive `highlights`, so there's no direct
extractive label — we approximate one instead (a standard technique for
deriving extractive labels from abstractive-summary datasets, used e.g. in
BERTSUM/PreSumm).

In [1]:
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from src.notebook_setup import setup_project

setup_project()

import datasets
import pandas as pd
import yaml
from datasets import load_dataset
from nltk.tokenize import sent_tokenize
from rouge_score import rouge_scorer
from transformers import AutoTokenizer


project root: /Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall
.env loaded: success ✅
NVIDIA_NIM_API_KEY: set ✅


/Users/lucyroh/Desktop/STUDY/Data Projects/rehearse-then-recall/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load `cnn_dailymail`

`cnn_dailymail` (config `"3.0.0"`) ships `article`/`highlights`/`id` fields
and official `train`/`validation`/`test` splits — unlike the AIHub pilot
(a single file needing a manual random split), we can just slice the splits
we need directly.

Three-way split, not just train/val: `validation` is used during training
for model selection (`05_rehearsal_maintenance_train.ipynb`'s
`load_best_model_at_end`/`metric_for_best_model="rougeL"`), so reporting
final metrics on that same split would be mildly optimistic (the checkpoint
was picked *because* it did well there). `test` is held out from training
entirely and only touched once, for the final reported numbers.

In [2]:
MAX_TRAIN_EXAMPLES = 3000
MAX_VAL_EXAMPLES = 300
MAX_TEST_EXAMPLES = 300

train_raw = load_dataset("cnn_dailymail", "3.0.0", split=f"train[:{MAX_TRAIN_EXAMPLES}]")
val_raw = load_dataset("cnn_dailymail", "3.0.0", split=f"validation[:{MAX_VAL_EXAMPLES}]")
test_raw = load_dataset("cnn_dailymail", "3.0.0", split=f"test[:{MAX_TEST_EXAMPLES}]")
print(f"train: {len(train_raw)}, validation: {len(val_raw)}, test: {len(test_raw)}")

sample = train_raw[0]
print("\nsample keys:", list(sample.keys()))
print("article (first 300 chars):", sample["article"][:300])
print("\nhighlights:", sample["highlights"])


train: 3000, validation: 300, test: 300

sample keys: ['article', 'highlights', 'id']
article (first 300 chars): LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappoi

highlights: Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .


## 2. Build (input, target) pairs via greedy ROUGE oracle extraction

Each article is first truncated to `configs/chunking.yaml`'s `max_words`
(350) — both to match real chunk-scale input (see intro) and so the oracle
extraction only ever picks sentences that are actually present in `input_text`
(picking from the *untruncated* article risked selecting a target sentence
that falls past the truncation point, which would train the model to produce
text it never saw — silently defeating the whole verbatim-extraction
objective). News articles are conventionally front-loaded ("inverted
pyramid"), so truncating to the first 350 words rarely drops the sentences
that actually matter — confirmed empirically: oracle extraction still
succeeded (returned at least one sentence) on 100/100 sampled articles after
truncation.

For each truncated article, we greedily add the sentence that most increases
the combined ROUGE-1/ROUGE-2 F-score against `highlights`, stopping once an
additional sentence's improvement drops below `MIN_GAIN` — without this
cutoff, the greedy search sometimes picks a near-irrelevant trailing sentence
just because it's marginally better than nothing (confirmed on a manual
example: a stray "Copyright 2007 Reuters." sentence got picked as a 3rd
choice without the cutoff).

In [3]:
CHUNK_MAX_WORDS = yaml.safe_load(open("configs/chunking.yaml", encoding="utf-8"))["max_words"]
MAX_ORACLE_SENTENCES = 3
MIN_GAIN = 0.01

_scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2"], use_stemmer=True)


def truncate_words(text: str, max_words: int) -> str:
    return " ".join(text.split()[:max_words])


def greedy_oracle_extract(sentences: list[str], reference_summary: str) -> list[int]:
    """Indices (ascending) of the sentences that best approximate an extractive
    summary of `reference_summary`, picked greedily by combined ROUGE-1/2 F1."""
    selected: list[int] = []
    selected_text = ""
    prev_score = 0.0
    remaining = set(range(len(sentences)))
    for _ in range(MAX_ORACLE_SENTENCES):
        best_idx, best_score = None, prev_score
        for i in remaining:
            candidate = (selected_text + " " + sentences[i]).strip()
            scores = _scorer.score(reference_summary, candidate)
            score = (scores["rouge1"].fmeasure + scores["rouge2"].fmeasure) / 2
            if score > best_score:
                best_idx, best_score = i, score
        if best_idx is None or (best_score - prev_score) < MIN_GAIN:
            break
        selected.append(best_idx)
        selected_text = (selected_text + " " + sentences[best_idx]).strip()
        prev_score = best_score
        remaining.discard(best_idx)
    return sorted(selected)


def build_pair(example: dict) -> dict | None:
    truncated = truncate_words(example["article"], CHUNK_MAX_WORDS)
    sentences = sent_tokenize(truncated)
    if not sentences:
        return None
    oracle_idx = greedy_oracle_extract(sentences, example["highlights"])
    if not oracle_idx:
        return None
    return {
        "id": example["id"],
        "input_text": " ".join(sentences),
        "target_text": " ".join(sentences[i] for i in oracle_idx),
    }


train_pairs = [p for p in (build_pair(ex) for ex in train_raw) if p is not None]
val_pairs = [p for p in (build_pair(ex) for ex in val_raw) if p is not None]
test_pairs = [p for p in (build_pair(ex) for ex in test_raw) if p is not None]
print(f"train pairs: {len(train_pairs)} / {len(train_raw)}")
print(f"val pairs: {len(val_pairs)} / {len(val_raw)}")
print(f"test pairs: {len(test_pairs)} / {len(test_raw)}")

pd.DataFrame(train_pairs)[["input_text", "target_text"]].head(3)


train pairs: 3000 / 3000
val pairs: 300 / 300
test pairs: 300 / 300


,input_text,target_text
0,"LONDON, England (Reuters) -- Harry Potter star...","Daniel Radcliffe as Harry Potter in ""Harry Pot..."
1,Editor's note: In our Behind the Scenes series...,"Here, Soledad O'Brien takes users inside a jai..."
2,"MINNEAPOLIS, Minnesota (CNN) -- Drivers who we...","""I probably had a 30-, 35-foot free fall."


## 3. Tokenize

`INPUT_MAX_LENGTH`/`TARGET_MAX_LENGTH` are set from measured token-length
percentiles on this data with the `t5-small` tokenizer (English SentencePiece,
~1.43 tokens/word — much lower than the old pko-t5 backbone's measured
~2.79 tokens/word): truncated (350-word) articles run ~487/529/569 tokens at
the 50th/90th/99th percentile, and oracle targets run ~72/99/127. 512 covers
the input up to ~90th percentile; 160 comfortably covers the target range.

In [4]:
INPUT_MAX_LENGTH = 512
TARGET_MAX_LENGTH = 160

tokenizer = AutoTokenizer.from_pretrained("t5-small")


def tokenize_pairs(pairs: list[dict]) -> datasets.Dataset:
    inputs = tokenizer(
        [p["input_text"] for p in pairs],
        max_length=INPUT_MAX_LENGTH,
        truncation=True,
    )
    targets = tokenizer(
        [p["target_text"] for p in pairs],
        max_length=TARGET_MAX_LENGTH,
        truncation=True,
    )
    return datasets.Dataset.from_dict(
        {
            "input_ids": inputs["input_ids"],
            "attention_mask": inputs["attention_mask"],
            "labels": targets["input_ids"],
        }
    )


train_dataset = tokenize_pairs(train_pairs)
val_dataset = tokenize_pairs(val_pairs)
test_dataset = tokenize_pairs(test_pairs)
print(train_dataset)


Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3000
})


## 4. Save

Save the tokenized datasets (train/val/test) and (for reproducibility/
debugging) the raw text pairs. `notebooks/05_rehearsal_maintenance_train.ipynb`
loads these directly — train+val during training, test only once at the end
for final reporting.

In [5]:
OUT_DIR = Path("data/processed/rehearsal_maintenance")
OUT_DIR.mkdir(parents=True, exist_ok=True)

train_dataset.save_to_disk(str(OUT_DIR / "train"))
val_dataset.save_to_disk(str(OUT_DIR / "val"))
test_dataset.save_to_disk(str(OUT_DIR / "test"))

pd.DataFrame(train_pairs).to_csv(OUT_DIR / "train_pairs_raw.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(val_pairs).to_csv(OUT_DIR / "val_pairs_raw.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(test_pairs).to_csv(OUT_DIR / "test_pairs_raw.csv", index=False, encoding="utf-8-sig")

print(f"Saved to: {OUT_DIR}")
print(f"  train_dataset: {len(train_dataset)} rows")
print(f"  val_dataset: {len(val_dataset)} rows")
print(f"  test_dataset: {len(test_dataset)} rows")


Saving the dataset (0/1 shards):   0%|          | 0/3000 [00:00<?, ? examples/s]

Saving the dataset (1/1 shards): 100%|██████████| 3000/3000 [00:00<00:00, 323826.13 examples/s]

Saving the dataset (1/1 shards): 100%|██████████| 3000/3000 [00:00<00:00, 307838.83 examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/300 [00:00<?, ? examples/s]

Saving the dataset (1/1 shards): 100%|██████████| 300/300 [00:00<00:00, 116778.77 examples/s]

Saving the dataset (1/1 shards): 100%|██████████| 300/300 [00:00<00:00, 104344.57 examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/300 [00:00<?, ? examples/s]

Saving the dataset (1/1 shards): 100%|██████████| 300/300 [00:00<00:00, 137563.27 examples/s]

Saving the dataset (1/1 shards): 100%|██████████| 300/300 [00:00<00:00, 121362.96 examples/s]

Saved to: data/processed/rehearsal_maintenance
  train_dataset: 3000 rows
  val_dataset: 300 rows
  test_dataset: 300 rows


## Summary

- Source: `cnn_dailymail` (`"3.0.0"`), official `train`/`validation`/`test`
  splits, sliced to `MAX_TRAIN_EXAMPLES`/`MAX_VAL_EXAMPLES`/`MAX_TEST_EXAMPLES`
  (3,000/300/300) for this pilot — no manual splitting needed (unlike the old
  AIHub single-file pilot). `test` is only used once, at the very end of
  `05_rehearsal_maintenance_train.ipynb`, for final reporting — never for
  model selection.
- Articles are truncated to `configs/chunking.yaml`'s `max_words` (350) before
  oracle extraction, both to match real chunk-scale input and to guarantee
  oracle-picked target sentences are always present in `input_text`.
- Extractive targets are *approximated* via greedy ROUGE-1/2 oracle
  extraction (`MAX_ORACLE_SENTENCES=3`, `MIN_GAIN=0.01`) — cnn_dailymail has
  no ground-truth extractive labels the way the old AIHub corpus did.
- `max_length` values (512 input / 160 target) come from measured token
  percentiles with the `t5-small` tokenizer (~1.43 tokens/word for English,
  vs. ~2.79 for the old pko-t5 backbone) — re-measure if the backbone or
  `configs/chunking.yaml`'s `max_words` changes.
- Next: scale up `MAX_TRAIN_EXAMPLES`/`MAX_VAL_EXAMPLES`/`MAX_TEST_EXAMPLES`
  once the pilot in `05_rehearsal_maintenance_train.ipynb` validates the code
  path end-to-end.